In [ ]:
!pip install tensorflow pandas numpy scikit-learn joblib

In [ ]:
import pandas as pd
import numpy as np
import joblib
import tensorflow as tf

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

print("TensorFlow version:", tf.__version__)
print("Libraries imported successfully!")

TensorFlow version: 2.20.0
Libraries imported successfully!


In [ ]:
data = {
    "food": [
        "Apple", "Banana", "Orange", "Rice", "Brown Rice",
        "Oats", "Egg", "Chicken Breast", "Paneer", "Milk",
        "Curd", "Dal", "Chickpeas", "Lentils", "Almonds",
        "Peanuts", "Potato", "Tomato", "Spinach", "Broccoli",
        "Avocado", "Bread", "Roti", "Pasta", "Fish"
    ],

    "category": [
        "Fruit", "Fruit", "Fruit", "Grain", "Grain",
        "Grain", "Protein", "Protein", "Dairy", "Dairy",
        "Dairy", "Protein", "Protein", "Protein", "Nuts",
        "Nuts", "Vegetable", "Vegetable", "Vegetable",
        "Vegetable", "Fruit", "Grain", "Grain", "Grain",
        "Protein"
    ],

    "calories": [
        52, 89, 47, 130, 123, 389, 155, 165, 265, 61,
        61, 116, 164, 116, 579, 567, 77, 18, 23, 34,
        160, 265, 297, 131, 206
    ],

    "protein": [
        0.3, 1.1, 0.9, 2.7, 2.7, 16.9, 13, 31, 18.3, 3.2,
        3.5, 9, 8.9, 9, 21.2, 25.8, 2, 0.9, 2.9, 2.8,
        2, 9, 11, 5, 22
    ],

    "carbs": [
        14, 22.8, 11.8, 28, 25.6, 66.3, 1.1, 0, 6.1, 4.8,
        4.7, 20, 27.4, 20, 21.6, 16.1, 17.5, 3.9, 3.6,
        7, 8.5, 49, 55, 25, 0
    ],

    "fat": [
        0.2, 0.3, 0.1, 0.3, 1, 6.9, 11, 3.6, 20.8, 3.3,
        3.3, 0.4, 2.6, 0.4, 49.9, 49.2, 0.1, 0.2, 0.4,
        0.4, 14.7, 3.2, 4.5, 1.1, 12
    ],

    "fiber": [
        2.4, 2.6, 2.4, 0.4, 1.6, 10.6, 0, 0, 0, 0,
        0, 7.9, 7.6, 7.9, 12.5, 8.5, 2.2, 1.2, 2.2,
        2.6, 6.7, 2.7, 11, 1.8, 0
    ]
}

df = pd.DataFrame(data)

df.head()

,food,category,calories,protein,carbs,fat,fiber
0,Apple,Fruit,52,0.3,14.0,0.2,2.4
1,Banana,Fruit,89,1.1,22.8,0.3,2.6
2,Orange,Fruit,47,0.9,11.8,0.1,2.4
3,Rice,Grain,130,2.7,28.0,0.3,0.4
4,Brown Rice,Grain,123,2.7,25.6,1.0,1.6


In [ ]:
df["nutrition_score"] = (
    df["protein"] * 2
    + df["fiber"] * 1.5
    - df["fat"] * 0.3
)

df[[
    "food",
    "protein",
    "fiber",
    "fat",
    "nutrition_score"
]].head(10)

,food,protein,fiber,fat,nutrition_score
0,Apple,0.3,2.4,0.2,4.14
1,Banana,1.1,2.6,0.3,6.01
2,Orange,0.9,2.4,0.1,5.37
3,Rice,2.7,0.4,0.3,5.91
4,Brown Rice,2.7,1.6,1.0,7.50
5,Oats,16.9,10.6,6.9,47.63
6,Egg,13.0,0.0,11.0,22.70
7,Chicken Breast,31.0,0.0,3.6,60.92
8,Paneer,18.3,0.0,20.8,30.36
9,Milk,3.2,0.0,3.3,5.41


In [ ]:
encoder = LabelEncoder()

df["category_encoded"] = encoder.fit_transform(
    df["category"]
)

print(df[["food", "category", "category_encoded"]].head())

         food category  category_encoded
0       Apple    Fruit                 1
1      Banana    Fruit                 1
2      Orange    Fruit                 1
3        Rice    Grain                 2
4  Brown Rice    Grain                 2


In [ ]:
features = [
    "calories",
    "protein",
    "carbs",
    "fat",
    "fiber",
    "category_encoded"
]

X = df[features].values
y = df["nutrition_score"].values

print("Input shape:", X.shape)
print("Output shape:", y.shape)

Input shape: (25, 6)
Output shape: (25,)


In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print(X_scaled[:5])

[[-0.80557883 -1.01039236 -0.26061288 -0.55098194 -0.35738937 -0.99615554]
 [-0.5563613  -0.91752542  0.26156229 -0.54353221 -0.30604033 -0.99615554]
 [-0.83925688 -0.94074215 -0.39115667 -0.55843167 -0.35738937 -0.99615554]
 [-0.28020133 -0.73179153  0.57012034 -0.54353221 -0.87087985 -0.38874362]
 [-0.32735059 -0.73179153  0.42770893 -0.49138411 -0.56278557 -0.38874362]]


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 20
Testing samples: 5


In [ ]:
model = Sequential([

    Dense(
        64,
        activation="relu",
        input_shape=(X_train.shape[1],)
    ),

    Dense(
        32,
        activation="relu"
    ),

    Dropout(0.2),

    Dense(
        16,
        activation="relu"
    ),

    Dense(
        1,
        activation="linear"
    )
])

model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,073 (12.00 KB)

 Trainable params: 3,073 (12.00 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

print("ANN compiled successfully!")

ANN compiled successfully!


In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

Epoch 1/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 221ms/step - loss: 884.0112 - mae: 23.1534 - val_loss: 696.5714 - val_mae: 21.1632
Epoch 2/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - loss: 879.2123 - mae: 23.0753 - val_loss: 692.9365 - val_mae: 21.0976
Epoch 3/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - loss: 874.5111 - mae: 23.0035 - val_loss: 689.8959 - val_mae: 21.0382
Epoch 4/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - loss: 871.7227 - mae: 22.9495 - val_loss: 686.8291 - val_mae: 20.9797
Epoch 5/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - loss: 872.0948 - mae: 22.9284 - val_loss: 683.9136 - val_mae: 20.9229
Epoch 6/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - loss: 865.0057 - mae: 22.8437 - val_loss: 680.8799 - val_mae: 20.8639
Epoch 7/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 862.8633 - mae: 22.7818 - val_loss: 677.9251 - val_mae: 20.8060
Epoch 8/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - loss: 861.5546 - mae: 22.7505 - val_loss: 675.3395 - val_mae: 20.7518
Epoch 9/100
2/2 ━━━━━━━

In [ ]:
loss, mae = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("Test Loss:", loss)
print("Mean Absolute Error:", mae)

Test Loss: 63.14225387573242
Mean Absolute Error: 6.925706386566162


In [ ]:
model.save("nutriscan_ann.keras")

joblib.dump(
    scaler,
    "nutriscan_scaler.pkl"
)

joblib.dump(
    encoder,
    "nutriscan_encoder.pkl"
)

print("NutriScan ANN model saved successfully!")

NutriScan ANN model saved successfully!


In [ ]:
def recommend_foods(top_n=5):

    results = []

    for _, food in df.iterrows():

        category_code = encoder.transform(
            [food["category"]]
        )[0]

        input_data = [[
            food["calories"],
            food["protein"],
            food["carbs"],
            food["fat"],
            food["fiber"],
            category_code
        ]]

        input_scaled = scaler.transform(input_data)

        prediction = model.predict(
            input_scaled,
            verbose=0
        )[0][0]

        results.append({
            "Food": food["food"],
            "Calories": food["calories"],
            "Protein": food["protein"],
            "Carbs": food["carbs"],
            "Fat": food["fat"],
            "Fiber": food["fiber"],
            "ANN Score": round(float(prediction), 2)
        })

    result_df = pd.DataFrame(results)

    result_df = result_df.sort_values(
        by="ANN Score",
        ascending=False
    )

    return result_df.head(top_n)

In [ ]:
recommendations = recommend_foods(5)

recommendations

,Food,Calories,Protein,Carbs,Fat,Fiber,ANN Score
7,Chicken Breast,165,31.0,0.0,3.6,0.0,52.42
15,Peanuts,567,25.8,16.1,49.2,8.5,49.39
14,Almonds,579,21.2,21.6,49.9,12.5,48.97
5,Oats,389,16.9,66.3,6.9,10.6,46.99
24,Fish,206,22.0,0.0,12.0,0.0,40.11


In [ ]:
def analyze_food(food_name):

    result = df[
        df["food"].str.lower() ==
        food_name.lower()
    ]

    if result.empty:

        print("Food not found.")

        print("\nAvailable foods:")
        print(", ".join(df["food"]))

        return

    food = result.iloc[0]

    category_code = encoder.transform(
        [food["category"]]
    )[0]

    input_data = [[
        food["calories"],
        food["protein"],
        food["carbs"],
        food["fat"],
        food["fiber"],
        category_code
    ]]

    input_scaled = scaler.transform(input_data)

    prediction = model.predict(
        input_scaled,
        verbose=0
    )[0][0]

    print("=" * 45)
    print("             NUTRISCAN AI")
    print("=" * 45)

    print("Food:", food["food"])
    print("Category:", food["category"])
    print("Calories:", food["calories"], "kcal")
    print("Protein:", food["protein"], "g")
    print("Carbohydrates:", food["carbs"], "g")
    print("Fat:", food["fat"], "g")
    print("Fiber:", food["fiber"], "g")

    print(
        "ANN Nutrition Score:",
        round(float(prediction), 2)
    )

    print("=" * 45)

In [ ]:
analyze_food("Apple")

             NUTRISCAN AI
Food: Apple
Category: Fruit
Calories: 52 kcal
Protein: 0.3 g
Carbohydrates: 14.0 g
Fat: 0.2 g
Fiber: 2.4 g
ANN Nutrition Score: 4.83


In [ ]:
analyze_food("Egg")

             NUTRISCAN AI
Food: Egg
Category: Protein
Calories: 155 kcal
Protein: 13.0 g
Carbohydrates: 1.1 g
Fat: 11.0 g
Fiber: 0.0 g
ANN Nutrition Score: 25.83


In [ ]:
analyze_food("Milk")

             NUTRISCAN AI
Food: Milk
Category: Dairy
Calories: 61 kcal
Protein: 3.2 g
Carbohydrates: 4.8 g
Fat: 3.3 g
Fiber: 0.0 g
ANN Nutrition Score: 6.5


In [ ]:
analyze_food("Bread")

             NUTRISCAN AI
Food: Bread
Category: Grain
Calories: 265 kcal
Protein: 9.0 g
Carbohydrates: 49.0 g
Fat: 3.2 g
Fiber: 2.7 g
ANN Nutrition Score: 21.9


In [ ]:
def calculate_bmi(weight, height):

    height_m = height / 100

    bmi = weight / (height_m ** 2)

    return round(bmi, 2)


In [ ]:
def calculate_calories(
    age,
    weight,
    height,
    gender,
    activity
):

    if gender.lower() == "male":

        bmr = (
            10 * weight
            + 6.25 * height
            - 5 * age
            + 5
        )

    else:

        bmr = (
            10 * weight
            + 6.25 * height
            - 5 * age
            - 161
        )

    activity_factors = {
        "sedentary": 1.2,
        "light": 1.375,
        "moderate": 1.55,
        "active": 1.725
    }

    factor = activity_factors.get(
        activity.lower(),
        1.2
    )

    return round(bmr * factor)

In [ ]:
age = 58
weight = 70
height = 175
gender = "Female"
activity = "moderate"

bmi = calculate_bmi(weight, height)

calories = calculate_calories(
    age,
    weight,
    height,
    gender,
    activity
)

print("========== NUTRISCAN AI ==========")
print("BMI:", bmi)
print("Estimated Daily Calories:", calories, "kcal/day")
print("===================================")

========== NUTRISCAN AI ==========
BMI: 22.86
Estimated Daily Calories: 2081 kcal/day
